## Parameters

In [27]:
from pathlib import Path

RAW_EXP_DIR = Path("../../tmp/exps-musique-val/")
EXP_DIR = Path("./exps/")

In [28]:
# DEVICE_CONFIGS = ["0", "1", "2", "3"]
# DEVICE_CONFIGS = ["0,1", "2,3"]
DEVICE_CONFIGS = ["0,1,2,3"]

In [29]:
common_params = {
    "params.model.path": [
        # Small models
        # "Qwen/Qwen2.5-1.5B-Instruct",
        "Qwen/Qwen2.5-7B-Instruct",
        "meta-llama/Llama-3.1-8B-Instruct",
        "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B",

        # Medium models
        # "Qwen/Qwen2.5-14B-Instruct",
        # "Qwen/Qwen2.5-32B-Instruct",
        # "Qwen/Qwen2.5-Coder-32B-Instruct",
        # "meta-llama/Llama-3.1-70B-Instruct",
        # "Qwen/QwQ-32B",

        # GRPO models
        "bdsaglam/Llama-3.1-8B-Instruct-ragent-grpo-musique-merged",  # Ragent-1
        "/home/baris/repos/verifiers/outputs/Llama-3.1-8B-Instruct-ragent-grpo-20250508_213215-merged",  # Ragent-4
        "/home/baris/repos/verifiers/outputs/Llama-3.1-8B-Instruct-ragent-grpo-20250520_080809-merged",  # Ragent-5
        "bdsaglam/Llama-3.1-8B-Instruct-ragent-grpo-20250526_110630", # Ragent-6
    ],
    "params.model.temperature": [
        0.5,
    ],
    "params.model.top_p": [
        0.95,
    ],
    "params.model.few_shot_prob": [
        0.0,
    ],
    "params.retriever.name": [
        "hybrid-tei",
    ],
    "params.retriever.top_k": [
        1,
    ],
    "params.repeat": [
        1,
        5,
    ],
    "params.run": [
        1,
        # 2,
        # 3
    ],
}

In [30]:
varying_params_list = [
    {
        "params.dataset.path": ["bdsaglam/musique"],
        "params.dataset.name": ["answerable"],
        "params.dataset.split": ["validation"],
    },
    # {
    #     "params.dataset.path": ["bdsaglam/hotpotqa-distractor"],
    #     "params.dataset.name": ["default"],
    #     "params.dataset.split": ['validation'],
    # },
]

## Setup

In [31]:
# AUTOGENERATED! DO NOT EDIT! File to edit: ../../nbs/dvc.experiment.ipynb.

# %% auto 0
__all__ = ["parse_params", "parse_metrics", "parse_experiment", "parse_experiments", "load_experiments"]

# %% ../../nbs/dvc.experiment.ipynb 3
import json
from typing import Generator


# %% ../../nbs/dvc.experiment.ipynb 4
def parse_params(record):
    params_node = record.get("data", {}).get("params", {})
    params = {}
    for k, v in params_node.items():
        params.update(v.get("data", {}))
    return params


def parse_metrics(record):
    metrics_node = record.get("data", {}).get("metrics", {})
    metrics = {}
    for k, v in metrics_node.items():
        metrics.update(v.get("data", {}))
    return metrics


def parse_experiment(record):
    return {
        "id": record["rev"],
        "name": record["name"],
        "params": parse_params(record),
        "metrics": parse_metrics(record),
    }


def parse_experiments(data: list[dict]) -> Generator[dict, None, None]:
    for node in data:
        if node.get("error"):
            continue
        commit = node.get("rev")
        if experiments := (node.get("experiments") or []):
            for experiment in experiments:
                for rev in experiment.get("revs") or []:
                    if not rev.get("error"):
                        yield {"commit": commit, **parse_experiment(rev)}
        else:
            yield {"commit": commit, **parse_experiment(node)}


def load_experiments(json_filepath):
    with open(json_filepath, "r") as f:
        data = json.load(f)
    return list(parse_experiments(data))


In [32]:
import itertools
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn

In [33]:
def sorted_tuple(x):
    return tuple(sorted(x))

In [34]:
def jprint(obj):
    print(json.dumps(obj, indent=2))

In [35]:
filepaths = list(RAW_EXP_DIR.glob("*.json"))
experiments = [exp for fp in filepaths for exp in load_experiments(fp)]
for exp in experiments:
    if 'mode' not in exp['params']['retriever']:
        exp['params']['retriever']['mode'] = 'new'
print(f"{len(experiments)} experiments")
jprint(next(iter(experiments), None))

6 experiments
{
  "commit": "workspace",
  "id": "workspace",
  "name": null,
  "params": {
    "dataset": {
      "path": "bdsaglam/musique-mini",
      "name": "answerable",
      "split": "validation"
    },
    "model": {
      "path": "bdsaglam/Llama-3.1-8B-Instruct-ragent-grpo-musique-merged",
      "temperature": 0.5,
      "top_p": 0.95,
      "few_shot_prob": 0.0
    },
    "retriever": {
      "name": "hybrid-tei",
      "top_k": 1,
      "mode": "new"
    },
    "repeat": 1,
    "run": 1,
    "devices": "1"
  },
  "metrics": {
    "exact_match": 0.29,
    "f1": 0.38924054112060663,
    "supporting.precision": 0.8011666666666667,
    "supporting.recall": 0.6169444444444444,
    "supporting.f1": 0.6783253968253967,
    "citation.precision": 0.3125,
    "citation.recall": 0.17416666666666666,
    "citation.f1": 0.21183333333333332
  }
}


In [36]:
df = pd.json_normalize(experiments)
if not df.empty:
    df = df[df['params.dataset.path'] == 'bdsaglam/musique'].reset_index(drop=True)
    df.drop(columns=['params.devices'], inplace=True)
    print(f"{len(df)} experiments before preprocessing")
    df.head()

4 experiments before preprocessing


In [37]:
param_cols = [col for col in df.columns if col.startswith("params.")]
metric_cols = [col for col in df.columns if col.startswith("metrics.")]

In [38]:
if len(df):
    df['params.model.temperature'] = df['params.model.temperature'].astype(float).map(lambda x: round(x, 2))
    df['params.model.top_p'] = df['params.model.top_p'].astype(float).map(lambda x: round(x, 2))

In [39]:
if len(df):
    df.dropna(subset=param_cols + metric_cols, inplace=True, how="any")
    df.drop_duplicates(subset=param_cols, inplace=True, keep='last')

    print(f"{len(df)} experiments after preprocessing")
df.head()

2 experiments after preprocessing


,commit,id,name,params.dataset.path,params.dataset.name,params.dataset.split,params.model.path,params.model.temperature,params.model.top_p,params.model.few_shot_prob,...,params.repeat,params.run,metrics.exact_match,metrics.f1,metrics.supporting.precision,metrics.supporting.recall,metrics.supporting.f1,metrics.citation.precision,metrics.citation.recall,metrics.citation.f1
2,0a5c0bcd1343966fc314a0f04630fe9482377413,8694307ba41a41da2a63356adc7d7d4a5c5c874e,prize-cows,bdsaglam/musique,answerable,validation,bdsaglam/Llama-3.1-8B-Instruct-ragent-grpo-202...,0.5,0.95,0.0,...,5,1,0.290000,0.389241,0.801167,0.616944,0.678325,0.312500,0.174167,0.211833
3,0a5c0bcd1343966fc314a0f04630fe9482377413,b850f02a1c97cd3d42e550eb5f92178172113767,dazed-stir,bdsaglam/musique,answerable,validation,bdsaglam/Llama-3.1-8B-Instruct-ragent-grpo-202...,0.5,0.95,0.0,...,1,1,0.254034,0.294531,0.825555,0.652565,0.705839,0.364364,0.344125,0.351939


In [40]:
for col in param_cols:
    values = list(df[col].unique())
    print(f"- {col}: {values}")
    print()

- params.dataset.path: ['bdsaglam/musique']

- params.dataset.name: ['answerable']

- params.dataset.split: ['validation']

- params.model.path: ['bdsaglam/Llama-3.1-8B-Instruct-ragent-grpo-20250526_110630']

- params.model.temperature: [0.5]

- params.model.top_p: [0.95]

- params.model.few_shot_prob: [0.0]

- params.retriever.name: ['hybrid-tei']

- params.retriever.top_k: [1]

- params.retriever.mode: ['new']

- params.repeat: [5, 1]

- params.run: [1]



In [41]:
EXP_DIR.mkdir(parents=True, exist_ok=True)

for _, row in df.iterrows():
    filepath = EXP_DIR / f"{row['name']}.json"
    with open(filepath, "w") as f:
        json.dump(dict(row), f)

## Setup remaining experiments

In [42]:
df = pd.DataFrame([json.loads(fp.read_text()) for fp in EXP_DIR.glob("*.json")])
len(df)

14

In [43]:
def produce_experiment_configs(common_params, varying_params):
    # Generate all possible combinations of parameters
    varying_params = {**common_params, **varying_params}
    keys = varying_params.keys()
    values = varying_params.values()
    for instance in itertools.product(*values):
        yield dict(zip(keys, instance))

In [44]:
def produce_all_experiment_configs(common_params: dict, varying_params_list: list[dict]):
    for params in varying_params_list:
        for exp_config in produce_experiment_configs(common_params, params):
            yield exp_config

In [45]:
exp_configs = list(produce_all_experiment_configs(common_params, varying_params_list))
target_params = list(exp_configs[0].keys())
print(f"{len(exp_configs)} experiment configurations")
print(target_params)

14 experiment configurations
['params.model.path', 'params.model.temperature', 'params.model.top_p', 'params.model.few_shot_prob', 'params.retriever.name', 'params.retriever.top_k', 'params.repeat', 'params.run', 'params.dataset.path', 'params.dataset.name', 'params.dataset.split']


In [46]:
def preprocess_config(config):
    return {k: round(v, 2) if isinstance(v, float) else v for k, v in config.items()}

In [47]:
if len(df):
    existing_configs = [preprocess_config(config) for config in df[target_params].to_dict(orient="records")]
    existing_configs[0]
else:
    existing_configs = []

print("Existing exps:", len(existing_configs))

Existing exps: 14


In [48]:
# find the missing configurations
missing_configs = [
    preprocess_config(dict(kv))
    for kv in list(
        {tuple(sorted(config.items())) for config in exp_configs}
        - {tuple(sorted(config.items())) for config in existing_configs}
    )
]
print(f"{len(missing_configs)} missing configurations")

2 missing configurations


In [49]:
def make_command(exp_config):
    lines = ["dvc exp run --queue"]
    for target_param in target_params:
        arg_name = target_param.split(".", 1)[-1]
        arg_value = exp_config[target_param]
        if "[" in str(arg_value):
            arg_value = f'"{arg_value}"'
        lines.append(f"-S {arg_name}='{arg_value}'")

    command = " \\\n    ".join(lines)
    return command

In [50]:
with open("run.sh", "w") as f:
    f.write("#!/bin/sh\n\n")
    for i,exp_config in enumerate(missing_configs):
        command = make_command(exp_config)
        devices = DEVICE_CONFIGS[i % len(DEVICE_CONFIGS)]
        command += f" \\\n    -S devices='\"{devices}\"'"
        f.write(command)
        f.write("\n\n")

In [51]:
next(iter(existing_configs), None)

{'params.model.path': '/home/baris/repos/verifiers/outputs/Llama-3.1-8B-Instruct-ragent-grpo-20250520_080809-merged',
 'params.model.temperature': 0.5,
 'params.model.top_p': 0.95,
 'params.model.few_shot_prob': 0.0,
 'params.retriever.name': 'hybrid-tei',
 'params.retriever.top_k': 1,
 'params.repeat': 1,
 'params.run': 1,
 'params.dataset.path': 'bdsaglam/musique',
 'params.dataset.name': 'answerable',
 'params.dataset.split': 'validation'}

In [52]:
next(iter(missing_configs), None)

{'params.dataset.name': 'answerable',
 'params.dataset.path': 'bdsaglam/musique',
 'params.dataset.split': 'validation',
 'params.model.few_shot_prob': 0.0,
 'params.model.path': 'deepseek-ai/DeepSeek-R1-0528-Qwen3-8B',
 'params.model.temperature': 0.5,
 'params.model.top_p': 0.95,
 'params.repeat': 5,
 'params.retriever.name': 'hybrid-tei',
 'params.retriever.top_k': 1,
 'params.run': 1}